# Project data cleaning verification

This notebook runs the authoritative Python cleaning script and reviews its audit, sample, and five record checks. Set `DSO576_DATA_DIR` to the directory containing the four unchanged source files before running all cells.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ['DSO576_DATA_DIR']).expanduser().resolve()
print('Project root:', PROJECT_ROOT.name)
print('Source files:', sorted(path.name for path in DATA_DIR.iterdir() if path.is_file() and not path.name.startswith('~$')))

Project root: dso576-project
Source files: ['BOS_M2025_dl.xlsx', 'MSA_M2025_dl.xlsx', 'Occupation Data (1).xlsx', 'aei_claude_ai_2026-06-26.csv']


In [2]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'src' / 'clean_project_data.py'),
    '--data-dir', str(DATA_DIR),
    '--output-root', str(PROJECT_ROOT),
]
completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)

Anthropic filtered rows: 7,294
AI state + soc6 rows: 6,607
BLS detailed rows: 186,557
Final rows: 186,557
Matched AI rows: 49,866
Unmatched AI rows: 136,691
Sample rows: 50
Record checks passed: 5/5
Audit: reports/cleaning_audit.json



In [3]:
audit = json.loads((PROJECT_ROOT / 'reports' / 'cleaning_audit.json').read_text())
sample = pd.read_csv(PROJECT_ROOT / 'data' / 'sample' / 'area_occupation_sample.csv')
checks = pd.read_csv(PROJECT_ROOT / 'reports' / 'record_checks.csv')

assert audit['final_merge']['bls_rows_before_merge'] == audit['final_merge']['final_rows']
assert audit['final_merge']['final_duplicate_key_rows'] == 0
assert len(sample) <= 50
assert checks['passed'].all()

summary = pd.Series({
    'final_rows': audit['final_merge']['final_rows'],
    'matched_rows': audit['final_merge']['matched_rows'],
    'unmatched_rows': audit['final_merge']['unmatched_bls_rows'],
    'sample_rows': len(sample),
    'record_checks_passed': int(checks['passed'].sum()),
})
display(summary)
display(checks)
display(sample.head(10))

final_rows              186557
matched_rows             49866
unmatched_rows          136691
sample_rows                 50
record_checks_passed         5
dtype: int64

,check_id,title,original,expected,actual,passed
0,1,Direct occupation mapping,US-CA; 27-1014.00; pct=0.03,Exact O*NET match; state=CA; soc6=27-1014; val...,title=Special Effects Artists and Animators; s...,True
1,2,Aggregate multiple O*NET extensions,US-CA 11-3031.00=0.13; 11-3031.01=0.10; 11-303...,claude_usage_pct=0.41; onet_occupation_count=3,claude_usage_pct=0.41; onet_occupation_count=3,True
2,3,Suppressed BLS employment,MSA AREA=10180; OCC_CODE=13-1031; TOT_EMP=**,TOT_EMP missing; employment_suppressed=True; r...,TOT_EMP=<NA>; employment_suppressed=True,True
3,4,Top-coded BLS annual wage,MSA AREA=10540; OCC_CODE=29-1215; A_MEDIAN=#,A_MEDIAN missing; wage_topcoded=True; row reta...,A_MEDIAN=<NA>; wage_topcoded=True,True
4,5,State + occupation cross-source lookup,MSA AREA=12540; CA; OCC_CODE=13-2011; Anthropi...,BLS values retained; claude_usage_pct=0.11; me...,state=CA; claude_usage_pct=0.11; merge=both,True


,area_category,AREA,AREA_TITLE,PRIM_STATE,OCC_CODE,OCC_TITLE,TOT_EMP_raw,TOT_EMP,employment_suppressed,A_MEDIAN_raw,A_MEDIAN,wage_unavailable,wage_topcoded,JOBS_1000,LOC_QUOTIENT,is_multistate_msa,claude_usage_pct,onet_occupation_count,ai_merge_status
0,metropolitan,10180,"Abilene, TX",TX,13-1031,"Claims Adjusters, Examiners, and Investigators",**,NaN,True,35580,35580.0,False,False,NaN,NaN,False,0.02,1.0,both
1,metropolitan,10380,"Aguadilla, PR",PR,15-1211,Computer Systems Analysts,**,NaN,True,72900,72900.0,False,False,NaN,NaN,False,1.54,1.0,both
2,metropolitan,10380,"Aguadilla, PR",PR,49-2022,Telecommunications Equipment Installers and Re...,**,NaN,True,27190,27190.0,False,False,NaN,NaN,False,NaN,NaN,left_only
3,metropolitan,10420,"Akron, OH",OH,19-2043,Hydrologists,**,NaN,True,99470,99470.0,False,False,NaN,NaN,False,NaN,NaN,left_only
4,metropolitan,10420,"Akron, OH",OH,25-3011,"Adult Basic Education, Adult Secondary Educati...",**,NaN,True,37440,37440.0,False,False,NaN,NaN,False,NaN,NaN,left_only
5,metropolitan,10420,"Akron, OH",OH,49-2095,"Electrical and Electronics Repairers, Powerhou...",**,NaN,True,64120,64120.0,False,False,NaN,NaN,False,NaN,NaN,left_only
6,metropolitan,10420,"Akron, OH",OH,49-2096,"Electronic Equipment Installers and Repairers,...",**,NaN,True,44230,44230.0,False,False,NaN,NaN,False,NaN,NaN,left_only
7,metropolitan,10500,"Albany, GA",GA,21-1023,Mental Health and Substance Abuse Social Workers,**,NaN,True,39980,39980.0,False,False,NaN,NaN,False,NaN,NaN,left_only
8,metropolitan,10540,"Albany, OR",OR,29-1215,Family Medicine Physicians,50,50.0,False,#,NaN,False,True,1.003,1.45,False,NaN,NaN,left_only
9,metropolitan,10580,"Albany-Schenectady-Troy, NY",NY,29-1151,Nurse Anesthetists,170,170.0,False,#,NaN,False,True,0.388,1.16,False,NaN,NaN,left_only
